In [1]:
import nest
import numpy as np
import matplotlib.pyplot as plt


              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.9.0
 Built: Nov  4 2025 17:52:39

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.



In [2]:
def encode_input_to_wave(initial_time, end_time, sampling_rate):
    spike_times = np.arange(initial_time, end_time, sampling_rate)
    return spike_times

class DataloaderXOR:
    def __init__(self, n_in, epochs, iterations_steps, resolution):
        self.n_in = n_in
        self.epochs = epochs
        self.iteration_steps = iterations_steps
        self.resolution = resolution

        self.sequences = ["00", "01", "10", "11"]
        self.sequence_weights = [0.5, 0.2, 0.2, 0.1]

        self.map_sequence_to_freq = {
            "0": 25,
            "1": 51
        }

        self.data, self.targets, self.pattern_history = self.generate_data()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, epoch):
        return self.data[epoch], self.targets[epoch]

    def generate_pattern(self):
        xor_pattern = generate_pattern()
        return xor_pattern

    def generate_data(self):
        data = []
        targets = []
        pattern_history = []
        for epoch in range(self.epochs):
            pattern_sequence = np.random.choice(
                self.sequences, 
                p=self.sequence_weights)

            first_input = encode_input_to_wave(
                initial_time=(epoch * self.iteration_steps * self.resolution) + self.resolution,
                end_time=(epoch+1) * self.iteration_steps * self.resolution,
                sampling_rate=self.map_sequence_to_freq[pattern_sequence[0]]
            )
            second_input = encode_input_to_wave(
                initial_time=(epoch * self.iteration_steps * self.resolution) + self.resolution,
                end_time=(epoch+1) * self.iteration_steps * self.resolution,
                sampling_rate=self.map_sequence_to_freq[pattern_sequence[1]]
            )
            
            if (pattern_sequence == "01") or (pattern_sequence == "10"):
                target = 1
            else:
                target = 0
                
            data.append([first_input, second_input])
            targets.append(target)
            pattern_history.append({
                "pattern": pattern_sequence,
            })
        return data, targets, pattern_history

In [3]:
params_setup = {
    "resolution": 0.1,
}

nest.ResetKernel()
nest.set(**params_setup)
nest.set_verbosity("M_FATAL")


May 29 15:51:40 SimulationManager::set_status [Info]: 
    Temporal resolution changed from 0.1 to 0.1 ms.


In [4]:
# Training parameters
epochs = 100
iteration_steps = 600

# Network parameters
n_in = 2  # number of input neurons
n_rec = 10  # number of recurrent neurons
n_out = 2  # number of readout neurons

In [5]:
mm_rec = nest.Create("multimeter", params={"record_from": ["V_m", "surrogate_gradient", "learning_signal"]})
mm_out = nest.Create("multimeter", params={"record_from": ["V_m", "readout_signal", "target_signal", "error_signal"]})
sr_in = nest.Create("spike_recorder")
sr_rec = nest.Create("spike_recorder")
wr = nest.Create("weight_recorder")

recorders = {
    "mm_rec": mm_rec,
    "mm_out": mm_out,
    "sr_in": sr_in,
    "sr_rec": sr_rec,
    "wr": wr
}

In [6]:
params_common_syn_eprop = {
    "optimizer": {
        "type": "gradient_descent",  # algorithm to optimize the weights
        "batch_size": 1,
        "optimize_each_step": False,  # call optimizer every time step (True) or once per spike (False); both
        # yield same results for gradient descent, False offers speed-up
        "Wmin": -100.0,  # pA, minimal limit of the synaptic weights
        "Wmax": 100.0,  # pA, maximal limit of the synaptic weights
    },
    "weight_recorder": wr,
}
nest.SetDefaults("eprop_synapse", params_common_syn_eprop)

In [7]:
class Network:
    def __init__(self, n_in, n_rec, n_out, delay, recorders):
        """
        Create network
            n_in (int)
            n_rec (int)
            n_out (int)
            recorders (dict): could contain keys sr_in, sr_rec, mm_rec, mm_out
        """
        # Network parameters
        self.n_in = n_in
        self.n_rec = n_rec
        self.n_out = n_out
        self.delay = delay # duration["step"]

        # Recorders
        self.recorders = recorders

        # Setup, creation and connection
        self.setup()
        self.create()
        self.connect()

        self.results_dict = {
            "error": [],
            "loss": [],
            "iteration": [],
            "label": [],
        }

    def setup(self):
        # Setup connection parameters
        self.params_conn_one_to_one = {
            "rule": "one_to_one"
        }
        self.params_conn_all_to_all = {
            "rule": "all_to_all", "allow_autapses": False
        }
        
        # Setup synaptic parameters
        self.params_syn_static = {
            "synapse_model": "static_synapse",
            "delay": self.delay,
        }
        self.params_syn_eprop = {
            "synapse_model": "eprop_synapse",
            "delay": self.delay,  # ms, dendritic delay
        }
        self.params_syn_feedback = {
            "synapse_model": "eprop_learning_signal_connection",
            "delay": self.delay
        }
        self.params_syn_learning_window = {
            "synapse_model": "rate_connection_delayed",
            "delay": self.delay,
            "receptor_type": 1,  # receptor type over which readout neuron receives learning window signal
        }
        self.params_syn_rate_target = {
            "synapse_model": "rate_connection_delayed",
            "delay": self.delay,
            "receptor_type": 2,  # receptor type over which readout neuron receives target signal
        }
    
    def create(self):
        """
        Network creation
        """
        # Spike input generator
        self.gen_spk_in = nest.Create("spike_generator", self.n_in)

        # Recurrent Network
        self.nrns_in = nest.Create("parrot_neuron", self.n_in)
        self.nrns_rec = nest.Create("eprop_iaf", self.n_rec)
        self.nrns_out = nest.Create("eprop_readout", self.n_out)

        # Output spike generators
        self.gen_rate_target = nest.Create("step_rate_generator", self.n_out)
        self.gen_learning_window = nest.Create("step_rate_generator")
        self.gen_spk_final_update = nest.Create("spike_generator", 1)

    def connect(self):
        self.connect_network()
        self.connect_recorders()
        
    def connect_network(self):
        """
        Connect network following connections on display
        """
        nest.Connect(
            self.gen_spk_in, 
            self.nrns_in, 
            self.params_conn_one_to_one, 
            self.params_syn_static)  # connection 1

        # Should this connections be sparse?? (2,3,4)
        
        nest.Connect(
            self.nrns_in, 
            self.nrns_rec,
            self.params_conn_all_to_all,
            self.params_syn_eprop)  # connection 2
        nest.Connect(
            self.nrns_rec, 
            self.nrns_rec,
            self.params_conn_all_to_all,
            self.params_syn_eprop)  # connection 3
        nest.Connect(
            self.nrns_rec, 
            self.nrns_out,
            self.params_conn_all_to_all,
            self.params_syn_eprop)  # connection 4
        
        nest.Connect(
            self.nrns_out, 
            self.nrns_rec, 
            self.params_conn_all_to_all, 
            self.params_syn_feedback)  # connection 5
        
        nest.Connect(
            self.gen_rate_target, 
            self.nrns_out, 
            self.params_conn_one_to_one, 
            self.params_syn_rate_target)  # connection 6
        nest.Connect(
            self.gen_learning_window, 
            self.nrns_out, 
            self.params_conn_all_to_all, 
            self.params_syn_learning_window)  # connection 7

        # Force final update to update all synapses 
        # it include all that have not being transmitted in the last update
        nest.Connect(
            self.gen_spk_final_update, 
            self.nrns_in + self.nrns_rec, 
            "all_to_all", {"weight": 1000.0})

    def connect_recorders(self):
        """
        Connect recorders
        """
        if self.recorders.get("sr_in"):
            nest.Connect(
                self.nrns_in, 
                self.recorders["sr_in"], 
                self.params_conn_all_to_all, 
                self.params_syn_static)
        if self.recorders.get("sr_rec"):
            nest.Connect(
                self.nrns_rec, 
                self.recorders["sr_rec"], 
                self.params_conn_all_to_all, 
                self.params_syn_static)
        if self.recorders.get("mm_rec"):
            nest.Connect(
                self.recorders["mm_rec"], 
                self.nrns_rec, 
                self.params_conn_all_to_all, 
                self.params_syn_static)
        if self.recorders.get("mm_out"):
            nest.Connect(
                self.recorders["mm_out"], 
                self.nrns_out, 
                self.params_conn_all_to_all, 
                self.params_syn_static)

    def evaluate(self, epoch, group_size, steps_sequence, steps_total_offset, steps_learning_window, resolution, phase_label):
        duration_sequence = steps_sequence * resolution
        duration_total_offset = steps_total_offset * resolution
        
        events_mm_out = self.recorders["mm_out"].get("events")
        
        readout_signal = events_mm_out["readout_signal"]
        target_signal = events_mm_out["target_signal"]
        senders = events_mm_out["senders"]
        times = events_mm_out["times"]

        print("readout_signal_prev", readout_signal.shape)

        cond1 = times > (epoch - 1) * group_size * duration_sequence + duration_total_offset
        cond2 = times <= epoch * group_size * duration_sequence + duration_total_offset
        idc = cond1 & cond2

        readout_signal = np.array([readout_signal[idc][senders[idc] == i] for i in set(senders)])
        target_signal = np.array([target_signal[idc][senders[idc] == i] for i in set(senders)])


        readout_signal = readout_signal.reshape((readout_signal.shape[0], 1, group_size, steps_sequence))
        target_signal = target_signal.reshape((target_signal.shape[0], 1, group_size, steps_sequence))

        readout_signal = readout_signal[:, :, :, -steps_learning_window :]
        target_signal = target_signal[:, :, :, -steps_learning_window :]

        loss = 0.5 * np.mean(np.sum((readout_signal - target_signal) ** 2, axis=3), axis=(0, 2))

        y_prediction = np.argmax(np.mean(readout_signal, axis=3), axis=0)
        y_target = np.argmax(np.mean(target_signal, axis=3), axis=0)
        accuracy = np.mean((y_target == y_prediction), axis=1)
        errors = 1.0 - accuracy

        self.results_dict["iteration"].append(epoch)
        self.results_dict["error"].extend(errors)
        self.results_dict["loss"].extend(loss)
        self.results_dict["label"].append(phase_label)

        

In [8]:
net = Network(n_in, n_rec, n_out, params_setup["resolution"], recorders)

In [9]:
data_loader = DataloaderXOR(
    n_in=n_in, 
    epochs=epochs, 
    iterations_steps=iteration_steps, 
    resolution=params_setup["resolution"])

In [10]:
group_size = 1

steps = {
    "learning_window": 10,
    "total_offset": 2,
    "sequence": iteration_steps
}

duration = {key: value * params_setup["resolution"] for key, value in steps.items()}

for epoch, (data, target) in enumerate(data_loader):
    print(epoch, data, target)
    one_hot_target = np.zeros(n_out)
    one_hot_target[target] = 1

    iteration_offset = epoch * group_size * duration["sequence"]
    
    params_gen_spk_in = [
        {"spike_times": data[0]},
        {"spike_times": data[1]},
    ]
    params_gen_rate_target = [
        {
            "amplitude_times": np.arange(
                0.0, 
                group_size * duration["sequence"], 
                duration["sequence"])
                + iteration_offset
                + duration["total_offset"],
            "amplitude_values": np.ones(1) if one_hot_target[i] else np.zeros(1)
        } for i in range(n_out)
    ]
    params_gen_learning_window = {
        "amplitude_times": np.hstack(
            [
                np.array([0.0, duration["sequence"] - duration["learning_window"]])
                + iteration_offset
                + group_element * duration["sequence"]
                + duration["total_offset"]
                for group_element in range(group_size)
            ]
        ),
        "amplitude_values": np.tile([0.0, 1.0], group_size)
    }
    
    nest.SetStatus(net.gen_spk_in, params_gen_spk_in)
    nest.SetStatus(net.gen_rate_target, params_gen_rate_target)
    nest.SetStatus(net.gen_learning_window, params_gen_learning_window)

    nest.Simulate(duration["total_offset"])
    if epoch > 0:
        print("params_gen_spk_in", params_gen_spk_in)
        net.evaluate(epoch, group_size, steps["sequence"], steps["total_offset"], steps["learning_window"], params_setup["resolution"], "train")
    nest.Simulate(group_size * duration["sequence"] - duration["total_offset"])

    
    if epoch == 3:
        break
        

0 [array([ 0.1, 25.1, 50.1]), array([ 0.1, 25.1, 50.1])] 0
1 [array([ 60.1, 111.1]), array([ 60.1, 111.1])] 0
params_gen_spk_in [{'spike_times': array([ 60.1, 111.1])}, {'spike_times': array([ 60.1, 111.1])}]
readout_signal_prev (120,)


ValueError: cannot reshape array of size 120 into shape (2,1,1,600)

In [ ]:
duration

In [ ]:
group_size = 1
input_group = data_loader[0][0]
target_group = [1]

params_gen_rate_target = [
    {
        "amplitude_times": np.arange(0.0, group_size * duration["sequence"], duration["sequence"])
        + iteration_offset
        + duration["total_offset"],
        "amplitude_values": np.zeros(group_size),
    }
    for _ in range(n_out)
]

In [ ]:
params_gen_rate_target

In [ ]:
params_gen_learning_window = {
    "amplitude_times": np.hstack(
        [
            np.array([0.0, 300 - 10])
            + iteration_offset
            + group_element * 300
            #+ 0.1
            for group_element in range(group_size)
        ]
    ),
    "amplitude_values": np.tile([0.0, 1.0], group_size),
}

In [ ]:
params_gen_learning_window

In [ ]:
nest.ResetKernel()

gen = nest.Create("spike_generator", params={"spike_times": [1, 2,3, 4, 5,6]})
nrn = nest.Create("eprop_readout")

mm = nest.Create("multimeter", params={"record_from": ["V_m", "readout_signal"]})

nest.Connect(mm, nrn)
nest.Connect(gen, nrn)

nest.Simulate(100)

mm.get()